# Encontro 5: RAG — Ingestão, Indexação e Consulta

Neste encontro, construímos um pipeline completo de Retrieval-Augmented Generation (RAG): ingestão e processamento de documentos, criação de embeddings, indexação em ChromaDB e consulta com recuperação semântica integrada ao LLM. Estrutura segue o estilo do Encontro 4.


In [ ]:
%pip install -q -r ../requirements.txt
%pip install -q chromadb langchain-text-splitters


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicialização do LLM — usar gemini-2.0-flash
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("MODEL_NAME", "gemini-2.0-flash")
assert api_key, "GOOGLE_API_KEY ausente. Defina no .env."
llm = ChatGoogleGenerativeAI(model=model_name, google_api_key=api_key, streaming=False)
print("LLM pronto (RAG).")


## 1) Arquitetura RAG

- Ingestão: coletar documentos brutos (PDFs, páginas web, markdown).
- Processamento: limpeza, normalização e "chunking" (dividir em trechos).
- Embeddings: transformar cada trecho em vetor semântico.
- Indexação: armazenar vetores em um banco de vetores (ChromaDB).
- Consulta: gerar pergunta → recuperar trechos relevantes → compor resposta com base no contexto recuperado (RAG).


## 2) Processamento de documentos (chunking)

Para RAG, dividimos documentos em "chunks" menores para melhor granularidade na recuperação. Usaremos `RecursiveCharacterTextSplitter`.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Exemplo simples: três documentos fictícios (poderiam ser carregados de arquivos)
docs_raw = [
    ("suporte_api", "Guia de Suporte de API:\n- Erros 401 indicam problemas de autenticação.\n- Use chaves de API válidas e verifique limites de taxa.\n- Logs detalhados ajudam a identificar falhas no deploy.\n"),
    ("deploy_guide", "Checklist de Deploy:\n1. Executar testes unitários e de integração.\n2. Validar variáveis de ambiente e secrets.\n3. Verificar rollback e monitoramento.\n"),
    ("faq_tickets", "FAQ de Tickets:\n- Prioridade alta para incidentes em produção.\n- Título até 140 caracteres, claro e objetivo.\n- Use tags para facilitar a triagem.\n")
]

docs = [Document(page_content=content, metadata={"source": src}) for src, content in docs_raw]
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Total de chunks: {len(chunks)}")
print("Exemplo de chunk:", chunks[0].page_content[:120].replace("\n", " "), "...")


## 3) Modelos de Embedding

Embeddings convertem texto em vetores semânticos. Critérios de escolha:
- Idioma e domínio dos seus documentos.
- Custo, latência e limites de uso.
- Dimensionalidade e qualidade (avaliar via métricas/benchmarks).

Aqui usamos embeddings da mesma plataforma do LLM para integração simples.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Inicializa embeddings (modelo de embeddings recomendado)
embeddings = GoogleGenerativeAIEmbeddings(google_api_key=api_key, model="models/text-embedding-004")
vec = embeddings.embed_query("Exemplo de texto para vetor semântico.")
print("Embedding gerado, dimensão:", len(vec))


## 4) Integração com ChromaDB

ChromaDB é um banco de vetores local/embarcado, ótimo para protótipos e pequenos projetos. Vamos indexar nossos chunks e criar um retriever por similaridade semântica.


In [ ]:
from langchain_community.vectorstores import Chroma

persist_dir = "./chroma_rag"
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=persist_dir)
vectorstore.persist()
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Index criado e persistido em", persist_dir)


## 5) Consulta RAG: recuperar contexto e gerar resposta

Vamos montar um chain LCEL que: (1) recupera documentos via retriever; (2) formata o contexto; (3) insere no prompt; (4) gera a resposta com o LLM.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = PromptTemplate.from_template(
    "Você é um assistente especializado em suporte. Use estritamente o contexto fornecido.\n\nContexto:\n{context}\n\nPergunta: {question}"
)

# LCEL: retriever → format → prompt → llm → parse
# Usamos RunnablePassthrough para passar a pergunta intacta,
# e um RunnableLambda para garantir que o output seja string (m.content).
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | RunnableLambda(lambda m: getattr(m, "content", m))
    | StrOutputParser()
)
print("RAG chain pronto.")


## 6) Demo: consultando o índice

Faça perguntas relacionadas ao conteúdo dos documentos que ingerimos.


In [ ]:
pergunta = "Quais passos devo validar antes de um deploy?"
resp = rag_chain.invoke(pergunta)
print(resp)

# Outra pergunta
pergunta2 = "Quando um ticket deve ter prioridade alta?"
print(rag_chain.invoke(pergunta2))


## 6.1) Busca Lexical e Híbrida

Além da busca semântica por embeddings (Chroma), você pode testar:
- Busca lexical (BM25), que prioriza termos exatos.
- Busca híbrida (ensemble), combinando lexical e semântica para maior robustez.


In [ ]:
%pip install -q rank_bm25

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Configura retriever lexical (BM25) a partir dos chunks
lexical_retriever = BM25Retriever.from_documents(chunks)
lexical_retriever.k = 4

# Ensemble/híbrido: combina lexical (BM25) e semântico (Chroma)
hybrid_retriever = EnsembleRetriever(retrievers=[lexical_retriever, retriever], weights=[0.5, 0.5])

# Chains RAG específicos para cada tipo de busca
rag_chain_lexical = (
    {"context": lexical_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | RunnableLambda(lambda m: getattr(m, "content", m))
    | StrOutputParser()
)

rag_chain_hybrid = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | RunnableLambda(lambda m: getattr(m, "content", m))
    | StrOutputParser()
)

# Teste com mesma pergunta para comparar
pergunta_test = "Quais passos devo validar antes de um deploy?"
print("--- Lexical (BM25) ---")
print(rag_chain_lexical.invoke(pergunta_test))

print("--- Híbrida (Ensemble: BM25 + Chroma) ---")
print(rag_chain_hybrid.invoke(pergunta_test))

# Mostra fontes dos documentos recuperados em cada caso
docs_lex = lexical_retriever.get_relevant_documents(pergunta_test)
docs_hyb = hybrid_retriever.get_relevant_documents(pergunta_test)
print("Fontes (lexical):", [d.metadata.get("source") for d in docs_lex])
print("Fontes (híbrida):", [d.metadata.get("source") for d in docs_hyb])


## 7) Manutenção e Atualização do Índice

- Re-indexe ao adicionar/alterar documentos (execute novamente a etapa `from_documents`).
- Use `persist_directory` para manter o índice entre execuções.
- Ajuste `k` do retriever para mais/menos contexto.
- Avalie embeddings alternativos conforme seu domínio e idioma.
